In [14]:
import json 
from mksense.config import PROCESSED_DATA_DIR
repo = "scikit-learn"
doc_input_path = PROCESSED_DATA_DIR / repo / f"{repo}_docs_cleaned.json"
with open(doc_input_path, 'rt') as  f_in:
    documents = json.load(f_in)

In [16]:
documents[0]

{'content': '.. raw :: html <!-- Generated by generate_authors_table.py --> <div class="sk-authors-container"> <style> img.avatar {border-radius: 10px;} </style> <div> <a href=\'https://github.com/ArturoAmorQ\'><img src=\'https://avatars.githubusercontent.com/u/86408019?v=4\' class=\'avatar\' /></a> <br /> <p>Arturo Amor</p> </div> <div> <a href=\'https://github.com/lucyleeow\'><img src=\'https://avatars.githubusercontent.com/u/23182829?v=4\' class=\'avatar\' /></a> <br /> <p>Lucy Liu</p> </div> <div> <a href=\'https://github.com/marenwestermann\'><img src=\'https://avatars.githubusercontent.com/u/17019042?v=4\' class=\'avatar\' /></a> <br /> <p>Maren Westermann</p> </div> <div> <a href=\'https://github.com/Charlie-XIAO\'><img src=\'https://avatars.githubusercontent.com/u/108576690?v=4\' class=\'avatar\' /></a> <br /> <p>Yao Xiao</p> </div> </div>',
 'file_path': 'documentation_team.rst',
 'title': '.. raw :: html',
 'context': '',
 'extension': 'rst',
 'repo': 'scikit-learn'}

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

base_url=os.environ.get("BASE_URL")
api_key=os.environ.get("API_KEY")

llm_client = OpenAI(
    base_url=base_url,
    api_key=api_key,
)

In [19]:
def build_prompt(query, search_results):
    prompt_template = """
    Your are a vertern software developer. 
    Answer the QUESTION based on the CONTEXT from the documentation database. 
    Use only the facts from the CONTEXT when answering the QUESTION. 
    If the CONTEXT doesn't containt the answer output None.

    QUESTION:
    {question}

    CONTEXT:
    {context}
    """.strip()
    
    context = ""

    for doc in search_results:
        context = context + f"repo: {doc['repo']}\ntitle:{doc['title']}\ndocument: {doc['content']}\n\n"

    prompt = prompt_template.format(question=query, context=context).strip()
    
    return prompt 


In [20]:
def llm(prompt):
    responce = llm_client.chat.completions.create(
        model="qwen/qwen3-coder:free",
        messages=[{
            "role":"user",
            "content":prompt
        }]
    )
    return responce.choices[0].message.content

In [21]:
query = "how do I install scikit-learn"